In [1]:
# HARD RESET
!rm -rf /content/UIDAI-Hackathon-2026
%cd /content

# CLONE GITHUB REPO
!git clone https://github.com/Romit-M/UIDAI-Hackathon-2026.git
%cd UIDAI-Hackathon-2026


/content
Cloning into 'UIDAI-Hackathon-2026'...
remote: Enumerating objects: 267, done.
remote: Total 267 (delta 0), reused 0 (delta 0), pack-reused 267 (from 1)
Receiving objects: 100% (267/267), 379.19 MiB | 27.42 MiB/s, done.
Resolving deltas: 100% (148/148), done.
Updating files: 100% (29/29), done.
/content/UIDAI-Hackathon-2026


### **Helpers & Normalization**

In [2]:
# IMPORTS
!pip install rapidfuzz
from rapidfuzz import fuzz, process

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# HELPER FUNCTIONS

# ======================
# Get file names
# ======================
def get_filename(category, r):
    return f"api_data_aadhar_{category}_{r}.csv"


# ======================
# Data loader
# ======================
def load_data(input_csv):
    df = pd.read_csv(input_csv)

    return df


# ======================
# Cached fuzzy matcher
# ======================

def fuzzy_match(value, choices, threshold=90):
    key = (value, tuple(choices))
    if key in cache:
        return cache[key]

    match = process.extractOne(value, choices, scorer=fuzz.token_sort_ratio)
    result = match[0] if match and match[1] >= threshold else None
    cache[key] = result
    return result


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.1 MB/s eta 0:00:00


### **DATA CLEANING**

In [3]:
# ======================
# Text normalization
# ======================
import re

def normalize_text(s):
    if not isinstance(s, str):
        return s

    s = s.lower().strip()
    s = s.replace('&', 'and').replace(' ', '')

    # Remove punctuation
    s = re.sub(r'[^a-z]', '', s)
    #s = re.sub(' +', '', s)

    return s


# ======================
# Feature engineering
# ======================
def add_columns(df):
  df['pin_prefix'] = df['pincode'].astype(str).str[:3]
  df['valid_flag'] = 'Not Valid'

  return df


# ======================
# Consolidated pipeline
# ======================
def clean_pipeline(df):

  df = df.copy()

  # Normalize text
  for col in ['state', 'district']:
    df[col] = df[col].astype(str).apply(normalize_text)

  # Add new columns
  df = add_columns(df)

  return df


### **DATA VALIDATION**

In [4]:
# REFERENCE DATASET PREPARATION

# Loading reference dataset
reference_data = pd.read_csv("data/external/unique_state_district_mapping.csv")

# Apply normalization
for col in ['state', 'district', 'state_map', 'district_map']:
    reference_data[col] = reference_data[col].astype(str).apply(normalize_text)


# PRECOMPUTE REFERENCE LOOKUPS

state_to_state_map = dict(zip(reference_data['state'], reference_data['state_map']))
district_to_district_map = dict(zip(reference_data['district'], reference_data['district_map']))
district_map_to_state_map = dict(
    zip(reference_data['district_map'], reference_data['state_map'])
)

state_maps = reference_data['state_map'].unique().tolist()
states = reference_data['state'].unique().tolist()

# state → districts lookup (state-scoped)
state_to_districts = {}
for state, group in reference_data.groupby('state_map'):
    state_to_districts[state] = {
        'district': group['district'].unique().tolist(),
        'district_map': group['district_map'].unique().tolist()
    }


In [5]:

# Cache for fuzzy matching results to avoid redundant string comparisons
cache = {}

def validate_dataset(df):

    df = df.copy()


    # =============================================
    # PHASE 1: STATE MATCH
    # =============================================
    resolved_state = {}

    for val in df['state'].unique():
        # Priority: state_map
        match = fuzzy_match(val, state_maps)
        if match:
            resolved_state[val] = match
            continue

        # Fallback: state → state_map
        match = fuzzy_match(val, states)
        if match:
            resolved_state[val] = state_to_state_map.get(match)

    df['state'] = df['state'].map(resolved_state).fillna(df['state'])
    state_resolved_mask = df['state'].isin(state_maps)

    # =============================================
    # PHASE 2: STATE-SCOPED DISTRICT MATCH
    # =============================================
    resolved_district = {}

    for state_val in df.loc[state_resolved_mask, 'state'].unique():
        districts_ref = state_to_districts[state_val]

        rows = df[
            (df['state'] == state_val) &
            (df['district'].notna())
        ]['district'].unique()

        for dist in rows:
            # Priority: district_map
            match = fuzzy_match(dist, districts_ref['district_map'])
            if match:
                resolved_district[dist] = match
                continue

            # Fallback: district → district_map
            match = fuzzy_match(dist, districts_ref['district'])
            if match:
                resolved_district[dist] = district_to_district_map.get(match)

    df.loc[state_resolved_mask, 'district'] = (
        df.loc[state_resolved_mask, 'district']
        .map(resolved_district)
        .fillna(df.loc[state_resolved_mask, 'district'])
    )

    # =============================================
    # PHASE 3: COLUMN-SHIFT RECOVERY
    # =============================================
    unresolved = ~state_resolved_mask
    recovered = {}

    for val in df.loc[unresolved, 'state'].unique():
        # Priority: district_map
        match = fuzzy_match(val, reference_data['district_map'].unique().tolist())
        if match:
            recovered[val] = match
            continue

        # Fallback: district
        match = fuzzy_match(val, reference_data['district'].unique().tolist())
        if match:
            recovered[val] = district_to_district_map.get(match)

    recovery_mask = df['state'].isin(recovered)

    df.loc[recovery_mask, 'district'] = df.loc[recovery_mask, 'state'].map(recovered)
    df.loc[recovery_mask, 'state'] = (
        df.loc[recovery_mask, 'district']
        .map(district_map_to_state_map)
    )

    # =============================================
    # FINAL VALIDATION
    # =============================================
    df.loc[
        df['state'].isin(state_maps) &
        df['district'].isin(reference_data['district_map']),
        'valid_flag'
    ] = 'Valid'

    return df


In [6]:
# DATA LOADING -> CLEANING -> SAVING CLEANED DATA

# Import tqdm to track progress
!pip install tqdm
from tqdm.auto import tqdm

# Define file name and path collectives
categories = ['biometric', 'demographic', 'enrolment']
ranges = ['0_500000', '500000_1000000']

RAW_PATH = "data/raw/api_data_aadhar"
PROCESSED_PATH = "data/processed/api_data_aadhar"


# Data validation for each file in collective
for category in tqdm(categories, desc="Categories"):
    for r in tqdm(ranges, desc="Ranges", leave=False):

        # Load file
        filename = get_filename(category, r)
        file_path = f"{RAW_PATH}_{category}/{filename}"
        df = load_data(file_path)

        # Apply feature engineering and cleaning
        df = clean_pipeline(df)

        # Validate data
        df = validate_dataset(df)

        # Save cleaned file
        output_filename = filename.replace('.csv', '_cleaned.csv')
        output_path = f"{PROCESSED_PATH}_{category}/{output_filename}"
        df.to_csv(output_path, index=False)

        print(f"{output_filename} saved.")


Categories:   0%|          | 0/3 [00:00<?, ?it/s]

Ranges:   0%|          | 0/2 [00:00<?, ?it/s]

api_data_aadhar_biometric_0_500000_cleaned.csv saved.
api_data_aadhar_biometric_500000_1000000_cleaned.csv saved.


Ranges:   0%|          | 0/2 [00:00<?, ?it/s]

api_data_aadhar_demographic_0_500000_cleaned.csv saved.
api_data_aadhar_demographic_500000_1000000_cleaned.csv saved.


Ranges:   0%|          | 0/2 [00:00<?, ?it/s]

api_data_aadhar_enrolment_0_500000_cleaned.csv saved.
api_data_aadhar_enrolment_500000_1000000_cleaned.csv saved.


In [7]:
# PUSH CLEANED FILES TO REPO

from google.colab import userdata
pat = userdata.get('GitHubAccessToken')

!git config --global user.name "Romit-M"
!git config --global user.email "romitrmaity@gmail.com"

!git add .
!git commit -m "Added cleaned data files"
!git push https://{pat}@github.com/Romit-M/UIDAI-Hackathon-2026.git


[main 88072a4] Added cleaned data files
 6 files changed, 456405 insertions(+), 456405 deletions(-)
Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 23.30 MiB | 2.25 MiB/s, done.
Total 13 (delta 9), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (9/9), completed with 4 local objects.
To https://github.com/Romit-M/UIDAI-Hackathon-2026.git
   2f94066..88072a4  main -> main
